# GPU Performance Baseline — Multi-Target Regression

这是一个基线模型，使用 随机森林回归（RandomForestRegressor）包装在 MultiOutputRegressor 中。
同时预测功耗、延迟、能效和吞吐量四个目标。

**A 榜 $R^2$ 得分**：约为 0.591480

**关键限制：**
- 未做特征工程（直接使用原始特征）
- 未处理类别特征（简单Label Encoding）
- 未做对数变换
- 未分场景建模

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

print('Libraries loaded.')

Libraries loaded.


In [2]:
# Load data
train = pd.read_csv('train.csv')
test_a = pd.read_csv('A.csv')
test_b = pd.read_csv('B.csv')

print(f'Train:   {train.shape}')
print(f'Test A:  {test_a.shape}')
print(f'Test B:  {test_b.shape}')

Train:   (723, 24)
Test A:  (1245, 20)
Test B:  (1248, 20)


### 查看数据结构

In [3]:
train.head(3)

,model,gpu_type,lambda_qps,scenario,avg_prompt_tokens,avg_generation_tokens,num_requests,total_b_params,num_layers,hidden_size,...,memory_type,release_year,base_clock_mhz,boost_clock_mhz,transistor_count_m,architecture,gpu_power_draw_watts,avg_e2e_latency_seconds,energy_efficiency_tokens_per_joule,throughput_tokens_per_second
0,allenai/OLMoE-1B-7B-0924,NVIDIA L40S,0.017,server_low,14.666667,256.000,3,7.00,16,2048,...,GDDR6,2022,1110.0,2520.0,76300.0,Ada Lovelace,110.019165,1.054412,0.018066,1.986887
1,meta-llama/Llama-3.2-3B-Instruct,NVIDIA L40S,-1.000,offline,14.264000,186.155,1000,3.21,28,3072,...,GDDR6,2022,1110.0,2520.0,76300.0,Ada Lovelace,329.491000,16.812138,33.690743,11059.409030
2,nvidia/AceMath-1.5B-Instruct,NVIDIA A100-SXM4-40GB,0.017,server_low,13.125000,256.000,8,1.50,28,1536,...,HBM2e,2020,1095.0,1410.0,54200.0,Ampere,72.151180,1.076503,0.093162,6.717419


In [4]:
# 目标列
TARGET_COLS = [
    'gpu_power_draw_watts',
    'avg_e2e_latency_seconds',
    'energy_efficiency_tokens_per_joule',
    'throughput_tokens_per_second',
]

# 确认目标列存在
available_targets = [c for c in TARGET_COLS if c in train.columns]
print(f'Targets: {available_targets}')

# 特征列 = 所有列 - 目标列
feature_cols = [c for c in train.columns if c not in TARGET_COLS]
print(f'Features ({len(feature_cols)}): {feature_cols[:10]}...')

Targets: ['gpu_power_draw_watts', 'avg_e2e_latency_seconds', 'energy_efficiency_tokens_per_joule', 'throughput_tokens_per_second']
Features (20): ['model', 'gpu_type', 'lambda_qps', 'scenario', 'avg_prompt_tokens', 'avg_generation_tokens', 'num_requests', 'total_b_params', 'num_layers', 'hidden_size']...


### 数据预处理

In [5]:
# 准备数据
X_train = train[feature_cols].copy()
y_train = train[available_targets].copy()
X_a = test_a[[c for c in feature_cols if c in test_a.columns]].copy()
X_b = test_b[[c for c in feature_cols if c in test_b.columns]].copy()

# 统一列
common_cols = list(set(X_train.columns) & set(X_a.columns) & set(X_b.columns))
X_train = X_train[common_cols]
X_a = X_a[common_cols]
X_b = X_b[common_cols]
print(f'Common feature columns: {len(common_cols)}')

# 丢弃全NaN行
y_train = y_train.dropna(how='all')
valid_idx = y_train.index.intersection(X_train.dropna(how='all').index)
X_train = X_train.loc[valid_idx]
y_train = y_train.loc[valid_idx]
print(f'After cleaning: {len(X_train)} training samples')

Common feature columns: 20
After cleaning: 723 training samples


In [6]:
# 分离数值特征和类别特征
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric features:   {len(numeric_cols)}')
print(f'Categorical features: {len(categorical_cols)} -> {categorical_cols}')

Numeric features:   14
Categorical features: 6 -> ['gpu_type', 'memory_type', 'scenario', 'model_type', 'model', 'architecture']


In [7]:
# 构建预处理Pipeline
# 数值特征: 中位数填充 + 标准化
# 类别特征: 众数填充 + 标签编码

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', LabelEncoder()),  # Note: ColumnTransformer requires special handling
])

# 对于类别特征，使用 OrdinalEncoder (兼容ColumnTransformer)
from sklearn.preprocessing import OrdinalEncoder
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols),
])

print('Preprocessor ready.')

Preprocessor ready.


### 训练模型

使用 随机森林回归（RandomForestRegressor）包装在 MultiOutputRegressor 中。
这等价于为每个目标独立训练一个线性模型。

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# 构建 Pipeline（直接使用 RandomForestRegressor）
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 训练
print('Training Random Forest model...')
model.fit(X_train, y_train)
print('Training complete!')

Training Random Forest model...


Training complete!


In [9]:
# 训练集上的快速验证
from sklearn.metrics import mean_absolute_error, r2_score

y_pred_train = model.predict(X_train)

for i, target in enumerate(available_targets):
    mae = mean_absolute_error(y_train[target], y_pred_train[:, i])
    r2 = r2_score(y_train[target], y_pred_train[:, i])
    
    # WMAPE
    wmape = np.sum(np.abs(y_train[target] - y_pred_train[:, i])) / np.sum(np.abs(y_train[target])) * 100
    
    print(f'{target}:')
    print(f'  MAE: {mae:.4f}, R²: {r2:.4f}, WMAPE: {wmape:.2f}%')

gpu_power_draw_watts:
  MAE: 2.8569, R²: 0.9972, WMAPE: 1.57%
avg_e2e_latency_seconds:
  MAE: 1.2566, R²: 0.9868, WMAPE: 6.91%
energy_efficiency_tokens_per_joule:
  MAE: 0.3540, R²: 0.9942, WMAPE: 3.72%
throughput_tokens_per_second:
  MAE: 89.5892, R²: 0.9961, WMAPE: 3.43%


### 生成预测提交文件

In [10]:
# 预测A榜
pred_a = model.predict(X_a)
df_pred_a = pd.DataFrame(pred_a, columns=available_targets)
df_pred_a.to_csv('A_predict.csv', index=False)
print(f'A_predict.csv saved: {df_pred_a.shape}')
df_pred_a.head()

A_predict.csv saved: (1245, 4)


,gpu_power_draw_watts,avg_e2e_latency_seconds,energy_efficiency_tokens_per_joule,throughput_tokens_per_second
0,335.854201,12.456967,37.817250,12553.814686
1,94.989393,0.908726,0.855134,78.316759
2,96.812181,0.996380,0.026810,2.489125
3,121.452094,6.936288,0.044762,4.998560
4,147.266817,1.071915,0.129479,16.981337


In [11]:
# 预测B榜
pred_b = model.predict(X_b)
df_pred_b = pd.DataFrame(pred_b, columns=available_targets)
df_pred_b.to_csv('B_predict.csv', index=False)
print(f'B_predict.csv saved: {df_pred_b.shape}')
df_pred_b.head()

B_predict.csv saved: (1248, 4)


,gpu_power_draw_watts,avg_e2e_latency_seconds,energy_efficiency_tokens_per_joule,throughput_tokens_per_second
0,97.710042,0.223256,0.003640,0.196651
1,246.538430,6.596706,0.272740,64.433874
2,357.445247,31.190486,20.340053,7292.927107
3,260.789809,5.345823,140.386041,35781.182138
4,112.733769,1.196795,0.177984,19.822113


In [12]:
# 验证预测文件格式
print('Target ranges in training set:')
for target in available_targets:
    print(f'  {target}: [{y_train[target].min():.2f}, {y_train[target].max():.2f}]')

print('\nPrediction ranges (A):')
for target in available_targets:
    print(f'  {target}: [{df_pred_a[target].min():.2f}, {df_pred_a[target].max():.2f}]')

Target ranges in training set:
  gpu_power_draw_watts: [29.63, 395.45]
  avg_e2e_latency_seconds: [0.01, 348.16]
  energy_efficiency_tokens_per_joule: [0.00, 269.56]
  throughput_tokens_per_second: [0.01, 53022.68]

Prediction ranges (A):
  gpu_power_draw_watts: [29.99, 392.74]
  avg_e2e_latency_seconds: [0.06, 331.82]
  energy_efficiency_tokens_per_joule: [0.00, 233.34]
  throughput_tokens_per_second: [0.20, 45213.93]
